### Camada silver

- remover duplicados
- tratar nulos
- corrigir tipos
- padronizar colunas
- validações

#### bronze_df

In [0]:
display(spark.sql("show databases"))

In [0]:
display(spark.sql("SHOW TABLES IN projeto"))

In [0]:
silver_df = spark.table("workspace.projeto.bronze_flights")

#### Obsevação do schema

##### Os tipos estão corretos?


In [0]:
silver_df.printSchema()

In [0]:
silver_df.dtypes

##### Visualização e descrição

In [0]:
silver_df.show(5)

In [0]:
silver_df.describe().show()

In [0]:
silver_df.count()

##### Quais colunas existem?

In [0]:
print(silver_df.columns)

##### Existem valores nulos?

In [0]:
from pyspark.sql.functions import col

for c in silver_df.columns:
    print(c, silver_df.filter(col(c).isNull()).count())

#### Limpeza

##### Retirando os Nulls

In [0]:
linhas_total = silver_df.count()

In [0]:
bronze_original = silver_df

In [0]:
bronze_original.filter(col("destination").isNull()).show(20)

In [0]:
bronze_original.filter(col("delay").like("%airline%")).show(20)

"Vou remover as linhas que possuem NULL em colunas essenciais para identificar um voo (origin e destination) ou que representam registros inválidos encontrados na ingestão."

In [0]:
bronze_raw = silver_df

In [0]:
bronze_raw.filter(col("origin").isNull() & col("destination").isNull()).count()

In [0]:
silver_df=silver_df.filter(col("origin").isNotNull() & col("destination").isNotNull())

##### Ajustando tipos das colunas

In [0]:
silver_test = silver_df.withColumn("delay_int", col("delay").try_cast("int"))

In [0]:
silver_test = silver_test.withColumn("distance_int", col("distance").try_cast("int"))

In [0]:
from pyspark.sql.functions import col, sum, when

contagem_nulos = [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in silver_test.columns]
silver_test.select(contagem_nulos).show()

In [0]:
silver_df = silver_df \
    .withColumn("delay", col("delay").try_cast("int")) \
    .withColumn("distance", col("distance").try_cast("int"))

In [0]:
from pyspark.sql.functions import max
silver_df.select(max("distance")).take(1)

In [0]:
silver_df.select(max("delay")).take(1)

In [0]:
silver_df.filter(col("distance").isNull()).count()

In [0]:
silver_df.filter(col("delay").isNull()).count()

In [0]:
silver_df.dtypes

##### Remevendo duplicatas

In [0]:
linhas_total = silver_df.count()
linhas_distintas = silver_df.distinct().count()

print(linhas_total-linhas_distintas)

In [0]:
silver_df = silver_df.dropDuplicates()

##### padronização dos dados

In [0]:
silver_df.select("destination").distinct().show(truncate=False)

In [0]:
silver_df.select("origin").distinct().show(truncate=False)

In [0]:
silver_df.select("delay", "distance").summary("min", "mean", "max").show()

In [0]:
silver_df.select("delay").orderBy(col("delay").desc()).show(20)

In [0]:
silver_df.select("delay").orderBy(col("delay")).show(20)

In [0]:
silver_df.select("distance").orderBy(col("distance").asc()).show(20)

In [0]:
silver_df.select("date").summary("min","max").show()

A coluna original `date` armazena as informações temporárias agregadas em uma string de 8 dígitos:  
**`X₁X₂X₃X₄X₅X₆X₇X₈`**

Para a camada Silver, o dado foi decomposto da seguinte forma:
* **`X₁X₂` (Mês):** Vai de `01` até `12` (identifica o mês do voo).
* **`X₃X₄` (Dia):** Vai de `01` até `31` (identifica o dia do mês).
* **`X₅X₆` (Hora):** Vai de `00` até `23` (identifica as horas).
* **`X₇X₈` (Minuto):** Vai de `00` até `59` (identifica os minutos).

In [0]:
from pyspark.sql.functions import col, substring, concat, lit, to_date

silver_df = silver_df.withColumn("date_str", col("date").cast("string"))

silver_df = silver_df \
            .withColumn("mes", substring(col("date_str"), 1, 2)) \
            .withColumn("dia", substring(col("date_str"), 3, 2)) \
            .withColumn("hora", substring(col("date_str"), 5, 2)) \
            .withColumn("minuto", substring(col("date_str"), 7, 2)) 

In [0]:
silver_df.filter(~col("date_str").rlike("^[0-9]+$")).show()

In [0]:
silver_df = silver_df.filter(col("date_str").rlike("^[0-9]+$"))

In [0]:
silver_df = silver_df \
            .withColumn("date_flight", to_date(concat(lit("2026-"), col("mes"), lit("-"), col("dia")), "yyyy-MM-dd")) \
            .withColumn("time_flight", concat(col("hora"), lit(":"), col("minuto")))

silver_df = silver_df.drop("date_str", "mes", "dia", "hora", "minuto", "date")

##### validações

In [0]:
silver_df.printSchema()

In [0]:
for c in silver_df.columns:
    print(c, silver_df.filter(col(c).isNull()).count())

In [0]:
print(silver_df.count()-silver_df.distinct().count())

In [0]:
silver_df.show(20)

In [0]:
silver_df.count()

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("projeto.silver_flights")